##**Assignment 3 (2024/2): ML1**
**Safe to eat or deadly poison?**



This homework is a classification task to identify whether a mushroom is edible or poisonous.

This dataset includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family Mushroom drawn from The Audubon Society Field Guide to North American Mushrooms (1981).

Each species is identified as definitely edible, definitely poisonous, or of unknown edibility and not recommended. This latter class was combined with the poisonous one. The Guide clearly states that there is no simple rule for determining the credibility of a mushroom; no rule like "leaflets three, let it be'' for Poisonous Oak and Ivy.


Step 1. Load 'mushroom2020_dataset.csv' data from the “Attachment” (note: this data set has been preliminarily prepared.).

Step 2. Drop rows where the target (label) variable is missing.

Step 3. Drop the following variables:
'id','gill-attachment', 'gill-spacing', 'gill-size','gill-color-rate', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring-rate','stalk-color-below-ring-rate','veil-color-rate','veil-type'

Step 4. Examine the number of rows, the number of digits, and whether any are missing.

Step 5. Fill missing values by adding the mean for numeric variables and the mode for nominal variables.

Step 6. Convert the label variable e (edible) to 1 and p (poisonous) to 0 and check the quantity. class0: class1

Step 7. Convert the nominal variable to numeric using a dummy code with drop_first = True.

Step 8. Split train/test with 20% test, stratify, and seed = 2020.

Step 9. Create a Random Forest with GridSearch on training data with 5 CV, scoring='f1-weighted',
	'criterion':['gini','entropy'],
'max_depth': [2,3],
'min_samples_leaf':[2,5],
'N_estimators':[100],
'random_state': 2020

Step 10.  Predict the testing data set with classification_report.


**Complete class MushroomClassifier from given code template below.**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [2]:
DROP_COLS = [
    'id', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color-rate',
    'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring',
    'stalk-color-above-ring-rate', 'stalk-color-below-ring-rate',
    'veil-color-rate', 'veil-type',
]


class MushroomClassifier:
    def __init__(self, data_path):  # DO NOT modify this line
        self.data_path = data_path
        self.df = pd.read_csv(data_path)

    def _prepared_df(self):
        df = self.df.dropna(subset=['label'])
        return df.drop(columns=DROP_COLS)

    def _imputed_labeled(self):
        df = self._prepared_df().copy()
        num_cols = df.select_dtypes(include='number').columns
        cat_cols = [c for c in df.select_dtypes(exclude='number').columns if c != 'label']

        for col in num_cols:
            df[col] = df[col].fillna(df[col].mean())
        for col in cat_cols:
            df[col] = df[col].fillna(df[col].mode()[0])

        y = df['label'].map({'p': 0, 'e': 1})
        X = df.drop(columns=['label'])
        return X, y

    def _split_encoded(self):
        X, y = self._imputed_labeled()
        X = pd.get_dummies(X, drop_first=True)
        return train_test_split(
            X, y, test_size=0.2, random_state=2020, stratify=y
        )

    def _grid_search(self):
        X_train, X_test, y_train, y_test = self._split_encoded()
        param_grid = {
            'criterion': ['gini', 'entropy'],
            'max_depth': [2, 3],
            'min_samples_leaf': [2, 5],
            'n_estimators': [100],
            'random_state': [2020],
        }
        gs = GridSearchCV(
            RandomForestClassifier(),
            param_grid,
            cv=5,
            n_jobs=-1,
            scoring='f1_weighted',
        )
        gs.fit(X_train, y_train)
        return gs, X_test, y_test

    def Q1(self):  # DO NOT modify this line
        return int(self.df['gill-size'].isna().sum())

    def Q2(self):  # DO NOT modify this line
        df = self._prepared_df()
        return df.shape[0], df.shape[1]

    def Q3(self):  # DO NOT modify this line
        _, y = self._imputed_labeled()
        return int((y == 0).sum()), int((y == 1).sum())

    def Q4(self):  # DO NOT modify this line
        X_train, X_test, _, _ = self._split_encoded()
        return X_train.shape, X_test.shape

    def Q5(self):  # DO NOT modify this line
        gs, _, _ = self._grid_search()
        p = gs.best_params_
        return (
            p['criterion'],
            p['max_depth'],
            p['min_samples_leaf'],
            p['n_estimators'],
            p['random_state'],
        )

    def Q6(self):  # DO NOT modify this line
        gs, X_test, y_test = self._grid_search()
        pred = gs.predict(X_test)
        report = classification_report(y_test, pred, output_dict=True)
        return (
            round(report['0']['f1-score'], 2),
            round(report['1']['f1-score'], 2),
        )


Run the code below to test that your code can work.

In [3]:
hw = MushroomClassifier('mushroom2020_dataset.csv')

print(hw.Q1())
print(hw.Q2())
print(hw.Q3())
print(hw.Q4())
print(hw.Q5())
print(hw.Q6())

121
(5764, 12)
(3660, 2104)
(4611, 1153)
{'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 100, 'random_state': 2020}
0.98
